# Identifiability within a cell type limits transfer across cell types

This notebook recomputes the second cross-task result of the paper *How Far Can Linear Go? BiPert, a Unified
Framework for Single-Cell Perturbation Modelling* (Section 6, Table 7, and the paragraph "why different
guides are needed" in Appendix F).

**The question.** A perturbation can only be recognised in a new cell type if its effect is reproducible to
begin with. Is a perturbation that is *identifiable within* each cell type also the one that *transfers across*
cell types?

**The data.** An independent screen: CRISPR activation of 1,833 transcription factors (TFs) in fibroblasts and
in RPE-1 cells (Southard et al., Nature Genetics 2025). Each TF is activated by several guide RNAs. The bundle
holds, for each cell type, the mean expression profile of the cells carrying each guide (the authors' released
per-guide means), restricted to the guides and genes present in both cell types.

**The test.** The guides of each TF are split into two disjoint halves. The profile of a TF from one half
*identifies* the TF if, among the profiles of all 1,833 TFs from the other half, the same TF is the best match
(top 1, in both directions of the comparison). The test is run within each cell type and across cell types.

**Requirements:** numpy, pandas. **Runtime:** under a minute.

## Configuration

In [ ]:
from pathlib import Path

DATA_DIR = Path("./crispra_data")
# Google Colab: copy the crispra_data folder to a folder BiPert on your Drive, then uncomment the next two lines.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_DIR = Path("/content/drive/MyDrive/BiPert/crispra_data")

In [ ]:
import time

import numpy as np
import pandas as pd

T_START = time.time()
CHECKS = []

def check(section, claim, paper, recomputed, tol=0):
    ok = abs(recomputed - paper) <= tol + 1e-9
    CHECKS.append(dict(section=section, claim=claim, paper=paper, recomputed=round(float(recomputed), 4), OK="PASS" if ok else "FAIL"))

def show(section):
    display(pd.DataFrame([c for c in CHECKS if c["section"] == section]).drop(columns="section"))

## Data and guide split

A guide is named `<TF>_<sequence>`. For each TF with at least four guides, the guides are sorted by name; the
first half forms set 1 and the second half set 2 (an odd guide out is dropped). The profile of a TF in one set is
the mean over that set's guides.

In [ ]:
fib = np.load(DATA_DIR / "fibroblast.npz"); rpe = np.load(DATA_DIR / "RPE1.npz")
guides, genes = fib["guides"], fib["genes"]
assert (guides == rpe["guides"]).all() and (genes == rpe["genes"]).all()
X = {"fib": fib["X"], "rpe": rpe["X"]}
tf_of_guide = np.array([g.split("_")[0].upper() for g in guides])
TFS, H = [], {k: [] for k in ("fib1", "fib2", "rpe1", "rpe2")}
for t in np.unique(tf_of_guide):
    idx = np.flatnonzero(tf_of_guide == t); idx = idx[np.argsort(guides[idx])]
    if len(idx) < 4:
        continue
    h = len(idx) // 2; TFS.append(t)
    for cell in ("fib", "rpe"):
        H[cell + "1"].append(X[cell][idx[:h]].mean(0)); H[cell + "2"].append(X[cell][idx[h:2 * h]].mean(0))
H = {k: np.array(v, dtype=np.float64) for k, v in H.items()}; N = len(TFS)
print(f"{len(guides):,d} guides in both cell types | {len(genes):,d} genes | {N:,d} TFs with at least four guides")

## The identification test

Profiles are centred across TFs (which removes what all activations share) and across genes, then scaled to unit
length. `test(a, b)` correlates every TF of set `a` with every TF of set `b`. A TF is a **hit** when its own
profile in the other set is the best match both ways: no other TF of `b` matches it better, and it matches no
other TF of `a` better.

In [ ]:
def unit(M):
    M = M - M.mean(0); M = M - M.mean(1, keepdims=True)
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)

U = {k: unit(v) for k, v in H.items()}

def test(a, b):
    C = U[a] @ U[b].T; d = np.diag(C)
    rank = np.maximum((C > d[:, None]).sum(1), (C > d[None, :]).sum(0))
    return d, rank == 0

TESTS = {"within fibroblasts, different guides": ("fib1", "fib2"), "within RPE-1, different guides": ("rpe1", "rpe2"),
         "across, same guides (set 1)": ("fib1", "rpe1"), "across, same guides (set 2)": ("fib2", "rpe2"),
         "across, different guides (fibroblast 1, RPE-1 2)": ("fib1", "rpe2"), "across, different guides (fibroblast 2, RPE-1 1)": ("fib2", "rpe1")}
R = {name: test(a, b) for name, (a, b) in TESTS.items()}
summary = pd.DataFrame({name: dict(mean_correlation=d.mean(), TFs_identified=int(hit.sum())) for name, (d, hit) in R.items()}).T
display(summary.round(3))

## Table 7: identified within, identified across

*Within* a cell type: the test between the two guide sets of that cell type. *Across*: the test between a guide
set in fibroblasts and the **other** guide set in RPE-1 cells, counted as a success if either pairing succeeds.
The strongest 10% of TFs are those with the largest signal: the logarithm of the dot product of a TF's two
centred half-profiles (noise in disjoint guide sets is independent and cancels in the product), averaged over the
two cell types.

In [ ]:
SEC = "Table 7"
within_fib = R["within fibroblasts, different guides"][1]; within_rpe = R["within RPE-1, different guides"][1]
across = R["across, different guides (fibroblast 1, RPE-1 2)"][1] | R["across, different guides (fibroblast 2, RPE-1 1)"][1]
both = within_fib & within_rpe; one = within_fib ^ within_rpe; neither = ~within_fib & ~within_rpe

def centred(M):
    M = M - M.mean(0); return M - M.mean(1, keepdims=True)

signal = []
for cell in ("fib", "rpe"):
    dot = (centred(H[cell + "1"]) * centred(H[cell + "2"])).sum(1)
    signal.append(np.log(np.maximum(dot, np.quantile(dot[dot > 0], 0.01))))
strength = (signal[0] + signal[1]) / 2
decile = np.digitize(strength, np.quantile(strength, np.linspace(0, 1, 11)[1:-1]))
strong = decile == 9

table8 = pd.DataFrame({"All TFs": {name: f"{int((g & across).sum())} / {int(g.sum())}" for name, g in (("Both cell types", both), ("One cell type", one), ("Neither", neither))},
                       "Strongest 10%": {name: f"{int((g & across & strong).sum())} / {int((g & strong).sum())}" for name, g in (("Both cell types", both), ("One cell type", one), ("Neither", neither))}})
table8.index.name = "Identified within"; display(table8)
check(SEC, "number of TFs", 1833, N)
for name, g, (hit, n), (hit_s, n_s) in (("both cell types", both, (25, 33), (25, 33)), ("one cell type", one, (24, 80), (21, 60)), ("neither", neither, (8, 1720), (5, 91))):
    check(SEC, f"identified within {name}: number of TFs", n, g.sum())
    check(SEC, f"identified within {name}: identified across", hit, (g & across).sum())
    check(SEC, f"strongest 10%, within {name}: number of TFs", n_s, (g & strong).sum())
    check(SEC, f"strongest 10%, within {name}: identified across", hit_s, (g & across & strong).sum())
show(SEC)

## Appendix F: why different guides are needed

A guide also has effects of its own, unrelated to its target. Comparing the *same* guides across the two cell
types therefore agrees more, and identifies more TFs, than the TF's effect alone would.

In [ ]:
SEC = "Appendix F: guides"
same = [R["across, same guides (set 1)"], R["across, same guides (set 2)"]]
diff = [R["across, different guides (fibroblast 1, RPE-1 2)"], R["across, different guides (fibroblast 2, RPE-1 1)"]]
within = [R["within fibroblasts, different guides"], R["within RPE-1, different guides"]]
r_same = sorted(d.mean() for d, _ in same); r_within = sorted(d.mean() for d, _ in within)
n_same = sorted(int(h.sum()) for _, h in same); n_diff = sorted(int(h.sum()) for _, h in diff)
print(f"mean correlation: same guide across cell types {r_same[0]:.3f} to {r_same[1]:.3f} | different guides within a cell type {r_within[0]:.3f} to {r_within[1]:.3f}")
print(f"TFs identified across cell types: same guides {n_same[0]} to {n_same[1]} | different guides {n_diff[0]} to {n_diff[1]} | ratio about {np.mean(n_same) / np.mean(n_diff):.1f}")
for label, paper, value in (("same guides across cell types, mean correlation (low)", 0.060, r_same[0]), ("same guides across cell types, mean correlation (high)", 0.065, r_same[1]),
                            ("different guides within a cell type, mean correlation (low)", 0.035, r_within[0]), ("different guides within a cell type, mean correlation (high)", 0.039, r_within[1])):
    check(SEC, label, paper, value, tol=0.0005)
for label, paper, value in (("TFs identified with the same guides (low)", 153, n_same[0]), ("TFs identified with the same guides (high)", 160, n_same[1]),
                            ("TFs identified with different guides (low)", 36, n_diff[0]), ("TFs identified with different guides (high)", 46, n_diff[1])):
    check(SEC, label, paper, value)
show(SEC)

## Summary: every check

In [ ]:
SUMMARY = pd.DataFrame(CHECKS)
print(f"{(SUMMARY.OK == 'PASS').sum()} of {len(SUMMARY)} checks pass | runtime {(time.time() - T_START):.0f} s")
display(SUMMARY)